In [1]:
import copy
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image, ImageFilter
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from tqdm import tqdm
import cv2

# 시드 고정 (재현성 확보)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==========================================
# 🌐 데이터셋 경로 설정 (🚨 반드시 본인 환경에 맞게 수정하세요!)
# ==========================================
# [1] Colab 또는 Local에서 실행할 경우: 데이터가 저장된 폴더 경로를 입력하세요.
COLAB_LOCAL_DATA_PATH = './competition_dataset/NEU-DET_open'
BASE_DIR = COLAB_LOCAL_DATA_PATH
SAVE_DIR = './'


# 하위 폴더 경로 설정 (압축 푼 폴더 구조에 맞춰 필요시 수정)
TRAIN_DIR = os.path.join(BASE_DIR, 'train', 'images')
VAL_DIR = os.path.join(BASE_DIR, 'validation', 'images')
TEST_DIR = os.path.join(BASE_DIR, 'test', 'images')

print(f"📂 베이스 경로: {BASE_DIR}")
print(f"📂 저장 경로: {SAVE_DIR}")

📂 베이스 경로: ./competition_dataset/NEU-DET_open
📂 저장 경로: ./


# Data Loader

In [3]:
# ==========================================
# 1. 커스텀 데이터 증강 (모션 블러)
# ==========================================
IMG_SIZE = 192
class RandomConveyorBeltMotionBlur:
    def __init__(self, kernel_size: int = 21, p: float = 0.7):
        self.kernel_size = kernel_size
        self.p = p

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return img

        kernel = np.zeros((self.kernel_size, self.kernel_size), dtype=np.float32)
        kernel[self.kernel_size // 2, :] = 1.0 / self.kernel_size

        if cv2 is not None:
            img_np = np.array(img)
            blurred = cv2.filter2D(img_np, -1, kernel)
            return Image.fromarray(blurred)

        return img.filter(ImageFilter.Kernel((self.kernel_size, self.kernel_size), kernel.flatten()))

# ==========================================
# 2. 트랜스폼 및 Dataset 세팅
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),                          
    RandomConveyorBeltMotionBlur(kernel_size=21, p=0.7), # 훈련 시 노이즈 예방주사
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 캐글 제출용 Test Dataset 클래스 (파일명 추출용)
class TestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.img_names = sorted(
            f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))
        )

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, img_name

# Data Loaders
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

NUM_CLASSES = len(train_dataset.classes)
print(f"✅ 클래스 개수: {NUM_CLASSES}")

✅ 클래스 개수: 6


# Model Definition

In [4]:
# ==========================================
# 💡 모델 세팅
# ==========================================
inference = True
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.mps.is_available()
    else 'cpu'
) if not inference else torch.device('cpu')
print(f"✅ 현재 디바이스: {DEVICE}")

# load checkpoint
LOAD_CHECK_DIR = Path("./artifacts/student_mobilenetv3_small.pth")
checkpoint = torch.load(LOAD_CHECK_DIR)
class_names = checkpoint['class_names']
class_names

model = models.mobilenet_v3_small(weights=None)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(class_names))
model.load_state_dict(checkpoint["state_dict"])
model = model.to(DEVICE).eval()


✅ 현재 디바이스: cpu


In [5]:
# Quantization (Linear layer)
engine = torch.backends.quantized.supported_engines[0]
print(engine)
torch.backends.quantized.engine = engine

model = copy.deepcopy(model).cpu().eval()
model = torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

qnnpack


In [6]:
# ==========================================
# ⏱️ 추론 시간 측정 및 submission.csv 생성
# ==========================================

# Test 데이터 로더 준비
test_dataset = TestDataset(TEST_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

predictions = []
image_ids = []

print("🚀 추론을 시작합니다...")

# --- ⏱️ 시간 측정 시작 ---
start_time = time.time()

with torch.no_grad():
    for images, img_names in tqdm(test_loader, desc="Inference"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        predictions.extend(preds.cpu().numpy())
        image_ids.extend(img_names)

# --- ⏱️ 시간 측정 종료 ---
end_time = time.time()
total_inference_time = end_time - start_time

print(f"✅ 추론 완료! 총 소요 시간: {total_inference_time:.2f}초")

# ==========================================
# 📝 제출용 CSV 저장
# ==========================================
inference_col = [""] * len(image_ids)
if inference_col:
    inference_col[0] = round(total_inference_time, 2)
submission_df = pd.DataFrame({
    'Id': image_ids,
    'Expected': predictions,
    'inference_time_sec': inference_col,
})

submission_path = os.path.join(SAVE_DIR, 'submission.csv')
submission_df.to_csv(submission_path, index=False)

print(f"🎉 제출 파일 저장 완료: {submission_path}")
submission_df.head()

🚀 추론을 시작합니다...


Inference: 100%|██████████| 6/6 [00:01<00:00,  3.66it/s]


✅ 추론 완료! 총 소요 시간: 1.65초
🎉 제출 파일 저장 완료: ./submission.csv


,Id,Expected,inference_time_sec
0,test_001.jpg,0,1.65
1,test_002.jpg,0,
2,test_003.jpg,0,
3,test_004.jpg,0,
4,test_005.jpg,0,
